# 📈 Crypto Day Trade (BTC · ETH · XRP) — รัน Streamlit บน Google Colab
ขั้นตอน: **1)** ตั้งค่า GitLab → **2)** Clone โค้ด → **3)** ติดตั้งไลบรารี → **4)** (ทางเลือก) อัปเดตข้อมูลแล้ว push กลับ GitLab → **5)** รันแอปและเปิดลิงก์

> ⚠️ เพื่อการศึกษาเท่านั้น ไม่ใช่คำแนะนำการลงทุน

## 1) ตั้งค่า GitLab

In [ ]:
import os, getpass

GITLAB_HOST    = "gitlab.com"                      # หรือ GitLab ขององค์กร เช่น gitlab.mycompany.com
GITLAB_PROJECT = "your-group/crypto-daytrade"      # ✏️ แก้เป็น group/repo ของคุณ
BRANCH         = "main"
DATA_SOURCE    = "gitlab"   # gitlab = แอปอ่าน CSV ผ่าน GitLab API | local = อ่านจากโฟลเดอร์ที่ clone | yfinance = ดึงสด

# ใส่ token เฉพาะกรณี repo เป็น private (สิทธิ์ read_repository + read_api; ถ้าจะ push ต้องมี write_repository)
token = getpass.getpass("GitLab Access Token (กด Enter ถ้า repo เป็น public): ").strip()

os.environ.update({
    "GITLAB_URL": f"https://{GITLAB_HOST}",
    "GITLAB_PROJECT": GITLAB_PROJECT,
    "GITLAB_BRANCH": BRANCH,
    "GITLAB_TOKEN": token,
    "DATA_SOURCE": DATA_SOURCE,
})
print("✔ ตั้งค่าเรียบร้อย")

## 2) Clone โค้ดจาก GitLab

In [ ]:
import subprocess
APP_DIR = "/content/app"
auth = f"oauth2:{token}@" if token else ""
clone_url = f"https://{auth}{GITLAB_HOST}/{GITLAB_PROJECT}.git"

subprocess.run(["rm", "-rf", APP_DIR])
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, clone_url, APP_DIR],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(r.stderr.replace(token, "***") if token else r.stderr)

# ลบ token ออกจาก remote URL ที่บันทึกในเครื่อง
subprocess.run(["git", "-C", APP_DIR, "remote", "set-url", "origin", f"https://{GITLAB_HOST}/{GITLAB_PROJECT}.git"])
os.chdir(APP_DIR)
!ls -la {APP_DIR} {APP_DIR}/data 2>/dev/null

## 3) ติดตั้งไลบรารี

In [ ]:
!pip install -q -r /content/app/requirements.txt

## 4) (ทางเลือก) อัปเดตข้อมูลล่าสุด แล้ว push กลับขึ้น GitLab
รันเซลล์นี้เมื่อต้องการเติมแท่งเทียนใหม่ลงใน `data/*.csv` (token ต้องมีสิทธิ์ `write_repository`)

In [ ]:
PUSH_TO_GITLAB = False   # ✏️ เปลี่ยนเป็น True ถ้าต้องการ push ไฟล์ข้อมูลใหม่ขึ้น GitLab

!python update_data.py --out data

if PUSH_TO_GITLAB:
    !git config user.email "colab@example.com" && git config user.name "colab-bot"
    !git add data/*.csv && git commit -q -m "update market data" || echo "ไม่มีการเปลี่ยนแปลง"
    push_url = f"https://oauth2:{token}@{GITLAB_HOST}/{GITLAB_PROJECT}.git"
    r = subprocess.run(["git", "push", "-q", push_url, f"HEAD:{BRANCH}"], capture_output=True, text=True)
    print("✔ push สำเร็จ" if r.returncode == 0 else r.stderr.replace(token, "***"))

## 5) รัน Streamlit และสร้างลิงก์ (Cloudflare Tunnel)
รอ 10–20 วินาที แล้วคลิกลิงก์ `https://....trycloudflare.com` · ลิงก์ใช้ได้ตราบที่ runtime ของ Colab ยังทำงาน

In [ ]:
import time, re

!wget -q -nc -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
!pkill -f "streamlit run" ; pkill -f cloudflared ; sleep 1

st_proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    cwd=APP_DIR, stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

cf_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
    stdout=open("/content/cloudflared.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        url = m.group(0); break
print("🌐 เปิดแอปที่:", url or "ยังไม่ได้ลิงก์ — ดู /content/cloudflared.log")

### ตรวจสอบ log (ถ้าแอปมีปัญหา)

In [ ]:
!tail -n 30 /content/streamlit.log